# 02 - Build Staging

Obiettivo di questo notebook: partire dalle 4 collection **source** (`sc_ram_ddr4_it`, `sc_ram_ddr4_de`, `sc_ram_ddr5_it`, `sc_ram_ddr5_de`), ognuna contenente 5 documenti (una pagina Idealo ciascuno, as-is), e produrre 4 collection **staging** pulite (`st_ram_ddr4_it`, `st_ram_ddr4_de`, `st_ram_ddr5_it`, `st_ram_ddr5_de`).

Cosa fa lo staging in questo notebook:
1. Flatten dei 5 documenti/pagina in una singola lista di prodotti per combinazione
2. Selezione dei campi utili e drop dei campi sempre-null (rumore)
3. Parsing di `characteristics` (lista di stringhe non strutturate, in italiano o tedesco) in campi numerici/strutturati
4. Deduplica sull'intera riga pulita (non solo su `id`, perché più offerte diverse possono condividere lo stesso `id` prodotto)
5. Report di profiling (null %, righe pre/post dedup) per ogni combinazione
6. Scrittura su MongoDB Atlas nelle collection di staging

**Nota:** in questo notebook non vengono ancora aggiunti i campi `market` e `ram_category` — verranno aggiunti nel notebook successivo, in fase di merge verso il DWH (`dw_ram_products`).

## 1. Setup e connessione a MongoDB

In [1]:
import os
import re
import json
from collections import Counter

import pandas as pd
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv()

MONGO_URI = os.getenv("MONGO_URI")

client = MongoClient(MONGO_URI)
db = client["idealo_ram"]

print("Collections disponibili:", db.list_collection_names())


Collections disponibili: ['dw_ram_products', 'st_ram_ddr5_de', 'sc_ram_ddr4_it', 'st_ram_ddr4_de', 'st_ram_ddr4_it', 'sc_ram_ddr4_de', 'sc_ram_ddr5_it', 'sc_ram_ddr5_de', 'st_ram_ddr5_it']


## 2. Mappatura combinazioni source -> staging

Le 4 combinazioni da processare. Ogni source collection contiene 5 documenti (uno per pagina), ognuno con la struttura Idealo grezza (`items`, `filterGroups`, `quickFilters`, ecc. — di cui a noi interessa solo `items`).

In [2]:
COMBINATIONS = [
    {"market": "IT", "category": "DDR4", "source": "sc_ram_ddr4_it", "staging": "st_ram_ddr4_it"},
    {"market": "IT", "category": "DDR5", "source": "sc_ram_ddr5_it", "staging": "st_ram_ddr5_it"},
    {"market": "DE", "category": "DDR4", "source": "sc_ram_ddr4_de", "staging": "st_ram_ddr4_de"},
    {"market": "DE", "category": "DDR5", "source": "sc_ram_ddr5_de", "staging": "st_ram_ddr5_de"},
]


## 3. Estrazione degli item grezzi

Legge i 5 documenti/pagina di una collection source e concatena i rispettivi `items` in un'unica lista.

In [3]:
def load_raw_items(collection_name):
    """Legge tutti i documenti (pagine) di una source collection e concatena i loro 'items'."""
    docs = list(db[collection_name].find({}))
    items = []
    for doc in docs:
        page_items = doc.get("items", [])
        items.extend(page_items)
    return items, len(docs)


## 4. Parsing di `characteristics`

`characteristics` è una lista di stringhe libere, non strutturate, con formattazione e lingua diverse tra IT e DE (es. `"32 GB"` vs `"32\xa0GB"`, `"Moduli 2"` vs `"Anzahl Module 2"`).

Le regex sotto sono pensate per essere **language-agnostic**: si basano sul pattern del valore (numero + unità), non sull'etichetta testuale che lo precede, così funzionano sia sui characteristics IT che DE senza bisogno di due logiche separate.

Per la latenza CAS teniamo due campi a granularità diversa, invece di comprimerli in uno solo:
- `cas_latency_raw`: sequenza completa dei timings se presente in `characteristics` (es. `"CL 16-16-16-36"`) — resta `None` quando non disponibile, che è un caso frequente (circa il 30-35% dei prodotti non la riporta affatto).
- `cas_latency_primary`: solo il primo numero CAS (es. `16`), più facile da confrontare tra prodotti. Se assente da `characteristics`, viene recuperato con un **fallback dal `title`** (spesso riporta `"CLxx"` anche quando `characteristics` non lo fa) — recupera circa metà dei casi altrimenti mancanti.

Non abbiamo aggiunto un campo che tracci la provenienza (`characteristics` vs fallback dal titolo): è ricavabile al volo confrontando i due campi (`cas_latency_raw` nullo + `cas_latency_primary` valorizzato = viene dal titolo), quindi tenerlo salvato sarebbe ridondanza evitabile anche in staging.

Se un pattern non trova corrispondenza, il campo derivato resta `None` — la lista originale viene comunque mantenuta in `characteristics_raw` per non perdere informazione e per poter affinare il parsing in seguito senza dover ripartire dal source.

In [4]:
def _norm(s):
    """Normalizza spazi (inclusi non-breaking space \xa0) e converte virgola decimale in punto."""
    return s.replace("\xa0", " ").strip()


def parse_characteristics(characteristics):
    """Estrae campi strutturati da una lista di stringhe characteristics (IT o DE)."""
    result = {
        "total_capacity_gb": None,
        "module_capacity_gb": None,
        "num_modules": None,
        "frequency_mts": None,
        "cas_latency_raw": None,
        "cas_latency_primary": None,
        "voltage_v": None,
        "form_factor": None,
    }

    if not characteristics:
        return result

    items = [_norm(c) for c in characteristics]

    for c in items:
        low = c.lower()

        # Capacita totale: stringa che e' SOLO "<numero> GB" (nessun prefisso testuale)
        m = re.fullmatch(r"(\d+)\s?GB", c)
        if m:
            result["total_capacity_gb"] = int(m.group(1))
            continue

        # Capacita per modulo: contiene "modul" ma anche "GB" con prefisso testuale
        m = re.search(r"(\d+)\s?GB", c)
        if m and "modul" in low and ("capacit" in low or "kapazit" in low):
            result["module_capacity_gb"] = int(m.group(1))
            continue

        # Numero moduli: contiene "modul" ma non "GB" (altrimenti sarebbe la capacita per modulo)
        m = re.search(r"(\d+)$", c)
        if m and "modul" in low and "gb" not in low:
            result["num_modules"] = int(m.group(1))
            continue

        # Frequenza: "<numero con . come separatore migliaia> MT/s"
        m = re.fullmatch(r"([\d\.]+)\s?MT/s", c)
        if m:
            result["frequency_mts"] = int(m.group(1).replace(".", ""))
            continue

        # Latenza CAS: sequenza completa tipo "CL 16-16-16-36" (a volte con prefisso testuale)
        m = re.search(r"CL\s?[\d]+(-[\d]+)+", c)
        if m:
            result["cas_latency_raw"] = m.group(0)
            first_num = re.search(r"CL\s?(\d+)", m.group(0))
            if first_num:
                result["cas_latency_primary"] = int(first_num.group(1))
            continue

        # Voltaggio: "<numero con , o . come decimale> V" a fine stringa
        # (ancorato solo a fine stringa, non a inizio: su IT spesso e' preceduto da "Voltaggio ")
        m = re.search(r"([\d,\.]+)\s?V$", c)
        if m:
            result["voltage_v"] = float(m.group(1).replace(",", "."))
            continue

        # Form factor
        if low in ("udimm", "sodimm", "so-dimm"):
            result["form_factor"] = c.upper().replace("-", "")
            continue

    return result


def extract_cas_primary_from_title(title):
    """Fallback: cerca 'CLxx' nel titolo del prodotto quando characteristics non riporta la latenza."""
    if not title:
        return None
    m = re.search(r"CL(\d+)", title, re.IGNORECASE)
    if m:
        return int(m.group(1))
    return None


## 5. Selezione e pulizia dei campi

Campi tenuti dal record grezzo (in base al profiling fatto su `sc_ram_*`):

- Identificativi: `id`, `offerId`, `title`
- Descrittivi: `subheading`, `characteristics` (parsati + raw)
- Recensioni: `userReviewCount`, `roundedRating` (spesso entrambi null insieme: assenza di recensioni, non un errore)
- Prezzo/offerta: `rawPrice`, `offerInfo.offerCount`, `offerInfo.usedOnly`
- Negozio: `shopInfo.shopName`, `shopInfo.shippingCosts`, `shopInfo.shippingIsFreeOfCharge`
- Altri: `isBestseller`, `hasProsAndCons`, `href`

Campi droppati perche' sempre (o quasi sempre) null nel profiling: `energyInfo`, `lastUpdated`, `retailMediaInfo`, `voucherCode`, `voucherDisclaimer`, `bestsellerInfo`, `formattedTestRating`, `bargainInfo`, `visualSearchItems`, `sources`, `categoryId` (costante), `imageUrl`, `shopInfo.deliveryInformation`, `shopInfo.deliveryStatus`, `shopInfo.merchantId`, `shopInfo.shippingCostsMap`, `shopInfo.freeReturnShipping`, `shopInfo.freeReturnDays`, `shopInfo.shippingComment`, `shopInfo.shopPageUrl`, `shopInfo.shopImageUrl`.

`mainProductId` viene invece **tenuto** (non droppato): su un campione di 180 prodotti, 7 avevano questo campo valorizzato, e collegava un `id` "child" a un `id` "parent" gia' presente nel dataset. Confrontando le coppie parent/child su piu' combinazioni pero' e' emerso che **non sono sempre lo stesso identico prodotto**: su DDR4 il pattern era coerente (stesso prezzo, stesso negozio sempre), ma su DDR5 circa un terzo delle coppie ha prezzo e/o negozio diversi — sono quindi prodotti/offerte realmente distinti (es. varianti collegate alla stessa scheda principale), non duplicati. La dedup nella sezione 7 droppa il parent solo quando prezzo e negozio coincidono esattamente con il child collegato.

**Da rivedere insieme:** questa selezione e' una prima proposta basata sul profiling fatto finora su un campione (108 item su 3 pagine) — va validata sull'intero dataset (720 prodotti) prima di considerarla definitiva.

In [5]:
def _parse_shipping_cost(raw):
    """'€ 8,19' -> 8.19 ; None se assente."""
    if not raw:
        return None
    m = re.search(r"([\d,\.]+)", raw)
    if not m:
        return None
    return float(m.group(1).replace(".", "").replace(",", "."))


def clean_item(raw_item):
    offer_info = raw_item.get("offerInfo") or {}
    shop_info = raw_item.get("shopInfo") or {}
    characteristics = raw_item.get("characteristics") or []
    title = raw_item.get("title")

    parsed_chars = parse_characteristics(characteristics)

    # Fallback: se la latenza CAS non e' riportata in characteristics, prova a recuperare
    # almeno il numero principale (es. "CL16") dal titolo del prodotto.
    if parsed_chars["cas_latency_primary"] is None:
        parsed_chars["cas_latency_primary"] = extract_cas_primary_from_title(title)

    cleaned = {
        "id": raw_item.get("id"),
        "offerId": raw_item.get("offerId"),
        "title": title,
        "subheading": raw_item.get("subheading"),
        "mainProductId": raw_item.get("mainProductId") or None,
        "characteristics_raw": tuple(characteristics),
        "userReviewCount": raw_item.get("userReviewCount"),
        "roundedRating": raw_item.get("roundedRating"),
        "rawPrice": raw_item.get("rawPrice"),
        "offerCount": offer_info.get("offerCount"),
        "usedOnly": offer_info.get("usedOnly"),
        "shopName": shop_info.get("shopName"),
        "shippingCosts": _parse_shipping_cost(shop_info.get("shippingCosts")),
        "shippingIsFreeOfCharge": shop_info.get("shippingIsFreeOfCharge"),
        "isBestseller": raw_item.get("isBestseller"),
    }
    cleaned.update(parsed_chars)
    return cleaned


## 6. Profiling pre/post pulizia

Piccolo report per combinazione: numero di righe raw, righe dopo dedup, e percentuale di null per i campi principali. Utile sia per validare la pulizia sia come base per la slide "dataset characteristics and quality issues".

In [6]:
def profile_dataframe(df, label):
    print(f"--- {label} ---")
    print(f"righe: {len(df)}")
    null_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
    print(null_pct[null_pct > 0])
    print()


## 7. Pipeline: per ogni combinazione -> estrai, pulisci, deduplica

Due livelli di deduplica, applicati in sequenza:

1. **Dedup esatta** sull'intera riga pulita. `characteristics_raw` e' una colonna di liste/tuple, non direttamente confrontabile da `drop_duplicates()` su alcuni tipi di valori non hashable: costruiamo quindi una chiave di confronto serializzando ogni riga in JSON, deduplichiamo su quella chiave, poi la scartiamo mantenendo le colonne originali.
2. **Dedup parent/child** tramite `mainProductId`: una riga "parent" viene scartata solo se e' un vero duplicato del "child" collegato (stesso `rawPrice` **e** stesso `shopName`) — se anche solo uno dei due differisce, sono prodotti/offerte realmente distinti e vengono mantenuti entrambi.

In [7]:
def dedup_full_row(df):
    """Deduplica sull'intera riga, gestendo anche colonne con valori non hashable (es. liste)."""
    dedup_key = df.apply(lambda row: json.dumps(row.to_dict(), sort_keys=True, default=str), axis=1)
    df = df.assign(_dedup_key=dedup_key)
    df = df.drop_duplicates(subset="_dedup_key").drop(columns="_dedup_key").reset_index(drop=True)
    return df


def remove_parent_duplicates(df):
    """Rimuove le righe 'parent' SOLO quando sono un vero duplicato del child collegato.

    Un child ha mainProductId valorizzato e punta all'id di un altro prodotto (il parent) gia'
    presente nel dataframe. Verificando i dati reali (DDR4 e DDR5), il collegamento mainProductId
    NON implica sempre che parent e child siano lo stesso identico annuncio: su DDR5 circa un
    terzo delle coppie ha prezzo e/o negozio diversi, quindi sono prodotti/offerte realmente
    distinti (es. varianti di capacita' o frequenza collegate alla stessa scheda principale),
    non un duplicato da rimuovere.

    Droppiamo il parent solo quando rawPrice E shopName coincidono ESATTAMENTE con quelli del
    child collegato: solo in quel caso e' un vero duplicato concettuale (stesso identico
    annuncio, il child ha semplicemente characteristics piu' complete). Se anche solo uno dei
    due differisce, entrambe le righe vengono mantenute.
    """
    by_id = df.set_index("id", drop=False)
    child_rows = df[df["mainProductId"].notna()]

    to_drop_ids = set()
    for _, child in child_rows.iterrows():
        parent_id = child["mainProductId"]
        if parent_id not in by_id.index:
            continue  # parent non presente in questa collection: nulla da droppare
        parent = by_id.loc[parent_id]
        same_price = parent["rawPrice"] == child["rawPrice"]
        same_shop = parent["shopName"] == child["shopName"]
        if same_price and same_shop:
            to_drop_ids.add(parent_id)

    to_drop = df["id"].isin(to_drop_ids)
    n_dropped = int(to_drop.sum())
    df_clean = df[~to_drop].reset_index(drop=True)
    return df_clean, n_dropped


staged_data = {}
dedup_stats = {}

for combo in COMBINATIONS:
    source_coll = combo["source"]
    staging_coll = combo["staging"]

    raw_items, n_pages = load_raw_items(source_coll)
    print(f"{source_coll}: {n_pages} pagine, {len(raw_items)} item grezzi")

    cleaned_items = [clean_item(it) for it in raw_items]
    df = pd.DataFrame(cleaned_items)

    n_before = len(df)
    df = dedup_full_row(df)
    n_after_exact = len(df)
    print(f"  dedup esatta: {n_before} -> {n_after_exact} righe ({n_before - n_after_exact} duplicati esatti rimossi)")

    df, n_parent_dropped = remove_parent_duplicates(df)
    n_after = len(df)
    print(f"  dedup parent/child: {n_after_exact} -> {n_after} righe ({n_parent_dropped} parent rimossi, child mantenuti)")

    profile_dataframe(df, staging_coll)

    staged_data[staging_coll] = df
    dedup_stats[staging_coll] = {
        "n_before": n_before,
        "n_after_exact": n_after_exact,
        "n_parent_dropped": n_parent_dropped,
        "n_after": n_after,
    }


sc_ram_ddr4_it: 5 pagine, 180 item grezzi
  dedup esatta: 180 -> 180 righe (0 duplicati esatti rimossi)
  dedup parent/child: 180 -> 174 righe (6 parent rimossi, child mantenuti)
--- st_ram_ddr4_it ---
righe: 174
mainProductId          96.0
userReviewCount        62.1
roundedRating          59.8
cas_latency_raw        27.0
cas_latency_primary    12.1
voltage_v               9.8
frequency_mts           2.9
num_modules             2.9
module_capacity_gb      2.9
form_factor             1.1
dtype: float64

sc_ram_ddr5_it: 5 pagine, 180 item grezzi
  dedup esatta: 180 -> 180 righe (0 duplicati esatti rimossi)
  dedup parent/child: 180 -> 152 righe (28 parent rimossi, child mantenuti)
--- st_ram_ddr5_it ---
righe: 152
userReviewCount        69.7
mainProductId          67.1
roundedRating          63.2
num_modules             2.6
module_capacity_gb      2.6
form_factor             2.0
voltage_v               2.0
cas_latency_raw         2.0
frequency_mts           1.3
cas_latency_primary     0

## 8. Controllo qualita' aggiuntivo: prodotti con `id` ripetuto

Come discusso, lo stesso `id` prodotto puo' comparire piu' volte con offerte (negozio/prezzo) diverse — non e' un duplicato da rimuovere, ma va tenuto d'occhio e documentato nella slide su data quality.

In [8]:
for staging_coll, df in staged_data.items():
    id_counts = df["id"].value_counts()
    repeated = id_counts[id_counts > 1]
    print(f"{staging_coll}: {len(repeated)} id con piu' di un'offerta (su {df['id'].nunique()} id univoci, {len(df)} righe totali)")


st_ram_ddr4_it: 0 id con piu' di un'offerta (su 174 id univoci, 174 righe totali)
st_ram_ddr5_it: 0 id con piu' di un'offerta (su 152 id univoci, 152 righe totali)
st_ram_ddr4_de: 0 id con piu' di un'offerta (su 174 id univoci, 174 righe totali)
st_ram_ddr5_de: 0 id con piu' di un'offerta (su 156 id univoci, 156 righe totali)


## 9. Data Quality Assessment

Dimensioni di qualita' calcolate sui dati puliti, ispirate alle dimensioni classiche viste anche negli esempi del corso (Accuracy, Completeness, Consistency, Uniqueness, Validity, Timeliness).

**Nota importante:** non tutte le dimensioni sono misurabili in modo oggettivo con i dati a disposizione:
- **Completeness**, **Uniqueness**, **Validity** e **Consistency** sono calcolate concretamente qui sotto, sui dati reali.
- **Accuracy** in senso stretto (i valori corrispondono alla realta'?) richiederebbe un riferimento esterno per la validazione (es. un catalogo ufficiale prodotti, come fa un progetto d'esempio del corso con un'ontologia dei comuni) che qui non abbiamo — la stiamo quindi trattata solo indirettamente tramite i controlli di *validity* (range plausibili) e *consistency* (coerenza tra campi correlati).
- **Timeliness/Currency** non sono applicabili in questo notebook: la raccolta e' un unico snapshot manuale (DevTools), non uno scraping incrementale schedulato — e' un limite del progetto da segnalare in slide, non un dato calcolabile qui.

In [9]:
def completeness_report(df, fields):
    """% di valori non-null per ciascun campo chiave."""
    pct = (df[fields].notna().mean() * 100).round(1)
    return pct.sort_values()


def uniqueness_report(df, staging_coll):
    stats = dedup_stats[staging_coll]
    id_counts = df["id"].value_counts()
    multi_offer_ids = (id_counts > 1).sum()
    return {
        "righe pre-dedup": stats["n_before"],
        "righe post dedup esatta": stats["n_after_exact"],
        "duplicati esatti rimossi": stats["n_before"] - stats["n_after_exact"],
        "parent rimossi (dedup parent/child)": stats["n_parent_dropped"],
        "righe finali": stats["n_after"],
        "id prodotto univoci": df["id"].nunique(),
        "id con piu' di un'offerta": multi_offer_ids,
    }


def validity_report(df):
    """Per ciascun campo numerico, % di valori (tra i non-null) che rientrano in un range plausibile."""
    checks = {}

    def pct_valid(series, condition_fn):
        valid = series.dropna()
        if len(valid) == 0:
            return None
        return round(condition_fn(valid).mean() * 100, 1)

    if "rawPrice" in df:
        checks["rawPrice > 0"] = pct_valid(df["rawPrice"], lambda s: s > 0)
    if "total_capacity_gb" in df:
        # Positiva e, quando module_capacity_gb e' noto, multipla di essa.
        # (non usiamo un elenco fisso di potenze di 2: DDR5 introduce densita' come 24/48GB
        # che sarebbero scartate a torto da un controllo pensato solo per DDR4)
        sub = df[df["total_capacity_gb"].notna()]
        if len(sub) > 0:
            positive = sub["total_capacity_gb"] > 0
            has_module = sub["module_capacity_gb"].notna()
            divisible = pd.Series(True, index=sub.index)
            divisible.loc[has_module] = (
                sub.loc[has_module, "total_capacity_gb"] % sub.loc[has_module, "module_capacity_gb"] == 0
            )
            checks["total_capacity_gb > 0 e multiplo di module_capacity_gb (se nota)"] = round(
                (positive & divisible).mean() * 100, 1
            )
    if "frequency_mts" in df:
        checks["frequency_mts in [1600, 8400]"] = pct_valid(
            df["frequency_mts"], lambda s: s.between(1600, 8400)
        )
    if "voltage_v" in df:
        checks["voltage_v in [1.0, 1.6]"] = pct_valid(
            df["voltage_v"], lambda s: s.between(1.0, 1.6)
        )
    if "num_modules" in df:
        checks["num_modules in [1, 8]"] = pct_valid(
            df["num_modules"], lambda s: s.between(1, 8)
        )

    return pd.Series(checks)


def consistency_report(df):
    """Verifica se total_capacity_gb == module_capacity_gb * num_modules, quando tutti e tre presenti."""
    mask = (
        df["total_capacity_gb"].notna()
        & df["module_capacity_gb"].notna()
        & df["num_modules"].notna()
    )
    subset = df[mask]
    if len(subset) == 0:
        return {"righe verificabili": 0, "coerenti (%)": None}

    consistent = subset["total_capacity_gb"] == (subset["module_capacity_gb"] * subset["num_modules"])
    return {
        "righe verificabili": len(subset),
        "righe totali": len(df),
        "coerenti (%)": round(consistent.mean() * 100, 1),
    }


In [10]:
KEY_FIELDS = [
    "id", "offerId", "title", "rawPrice", "shopName",
    "total_capacity_gb", "module_capacity_gb", "num_modules",
    "frequency_mts", "cas_latency_raw", "cas_latency_primary", "voltage_v", "form_factor",
]

for staging_coll, df in staged_data.items():
    print(f"=== {staging_coll} ===")

    print("-- Completeness (%) --")
    print(completeness_report(df, KEY_FIELDS))
    print()

    print("-- Uniqueness --")
    for k, v in uniqueness_report(df, staging_coll).items():
        print(f"  {k}: {v}")
    print()

    print("-- Validity (%) --")
    print(validity_report(df))
    print()

    print("-- Consistency (total = module x num_modules) --")
    for k, v in consistency_report(df).items():
        print(f"  {k}: {v}")
    print("\n")


=== st_ram_ddr4_it ===
-- Completeness (%) --
cas_latency_raw         73.0
cas_latency_primary     87.9
voltage_v               90.2
module_capacity_gb      97.1
num_modules             97.1
frequency_mts           97.1
form_factor             98.9
id                     100.0
offerId                100.0
title                  100.0
rawPrice               100.0
shopName               100.0
total_capacity_gb      100.0
dtype: float64

-- Uniqueness --
  righe pre-dedup: 180
  righe post dedup esatta: 180
  duplicati esatti rimossi: 0
  parent rimossi (dedup parent/child): 6
  righe finali: 174
  id prodotto univoci: 174
  id con piu' di un'offerta: 0

-- Validity (%) --
rawPrice > 0                                                        100.0
total_capacity_gb > 0 e multiplo di module_capacity_gb (se nota)    100.0
frequency_mts in [1600, 8400]                                       100.0
voltage_v in [1.0, 1.6]                                             100.0
num_modules in [1, 8]    

## 10. Scrittura su MongoDB (collection di staging)

**Nota tecnica importante:** pandas converte i valori mancanti in colonne miste (es. `mainProductId`, quasi sempre `None` ma con qualche stringa) in `NaN` (float), non in `None`. Se scritto cosi' com'e', pymongo salva un `NaN` letterale nel documento invece di `null` — cosa che rompe le query tipo `{"campo": {"$ne": None}}`, perche' un `NaN` e' tecnicamente diverso da `None`. Sanitizziamo quindi ogni `NaN` in `None` prima della scrittura.

In [11]:
import math

def sanitize_nans(records):
    """Sostituisce ogni NaN float (introdotto da pandas nei valori mancanti) con None vero."""
    for r in records:
        for k, v in r.items():
            if isinstance(v, float) and math.isnan(v):
                r[k] = None
    return records


for staging_coll, df in staged_data.items():
    df_to_write = df.copy()
    df_to_write["characteristics_raw"] = df_to_write["characteristics_raw"].apply(list)
    records = df_to_write.to_dict("records")
    records = sanitize_nans(records)

    db[staging_coll].delete_many({})  # idempotenza: puliamo prima di riscrivere
    if records:
        db[staging_coll].insert_many(records)

    print(f"{staging_coll}: {len(records)} documenti scritti")


st_ram_ddr4_it: 174 documenti scritti
st_ram_ddr5_it: 152 documenti scritti
st_ram_ddr4_de: 174 documenti scritti
st_ram_ddr5_de: 156 documenti scritti


## 11. Verifica finale

In [12]:
for combo in COMBINATIONS:
    staging_coll = combo["staging"]
    count = db[staging_coll].count_documents({})
    print(f"{staging_coll}: {count} documenti in MongoDB")


st_ram_ddr4_it: 174 documenti in MongoDB
st_ram_ddr5_it: 152 documenti in MongoDB
st_ram_ddr4_de: 174 documenti in MongoDB
st_ram_ddr5_de: 156 documenti in MongoDB
